# Notebook 06: Machine Learning Classification

**Project:** Cancer Microbiome Comparative Analysis  
**Description:** This notebook trains and evaluates multiple classifiers on microbiome data.  
**Tasks:**
1. Multi-class cancer-type classification (Colorectal vs Breast vs Prostate) — cancer samples only
2. Binary classification (Cancer vs Healthy) per cancer type (Colorectal and Breast)
3. Feature importance from Random Forest

**Inputs (from Results/):**
- `abund_combined_clr.csv` — CLR-transformed genus-level abundance
- `meta_combined.csv` — metadata with `cancer_type` and `condition`

**Outputs:**
- `Results/best_model_multiclass.pkl`
- `Results/model_comparison.csv`
- `Results/binary_classification_results.csv`
- `Results/rf_feature_importances.csv`
- `Figures/fig09_roc_curves.png`
- `Figures/fig10_confusion_matrix.png`
- `Figures/fig09b_binary_roc.png`
- `Figures/fig09c_rf_feature_importance.png`

In [ ]:
# Cell 1 — Imports
import pandas as pd    # pandas: tables and data manipulation (like Excel in Python)
import numpy as np     # numpy: fast math on arrays of numbers
import matplotlib      # matplotlib: core Python charting library
matplotlib.use('Agg')  # 'Agg' = save plots to files instead of opening pop-up windows
import matplotlib.pyplot as plt  # pyplot: easier interface for drawing charts
import seaborn as sns  # seaborn: prettier statistical charts built on matplotlib
import warnings        # warnings: controls whether Python shows warning messages
warnings.filterwarnings('ignore')  # silence harmless warnings
import os              # os: file and folder path utilities

# sklearn (scikit-learn): the main machine learning library for Python
from sklearn.ensemble import RandomForestClassifier   # Random Forest: many decision trees combined (ensemble)
from sklearn.linear_model import LogisticRegression   # Logistic Regression: linear model for classification
from sklearn.svm import SVC                           # SVM: Support Vector Machine — finds the best boundary between classes
from sklearn.preprocessing import LabelEncoder, label_binarize  # LabelEncoder: converts "Breast"/"Colorectal" to 0/1/2
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split  # tools for splitting data
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, auc)  # metrics to measure how well models perform
from sklearn.pipeline import Pipeline                # Pipeline: chains preprocessing + model into one step
from sklearn.preprocessing import StandardScaler     # StandardScaler: rescales features to mean=0, std=1 (required by SVM/LR)
import joblib  # joblib: saves trained models to files so they can be reused later

# XGBoost: a powerful tree-based model (common in competitions and industry)
try:
    from xgboost import XGBClassifier
    xgb_available = True
    print('XGBoost available.')
except ImportError:
    print('XGBoost not installed. Skipping XGBoost. pip install xgboost')
    xgb_available = False  # flag used in later cells to skip XGBoost if missing

# SMOTE: Synthetic Minority Oversampling — artificially creates new samples for the smaller class
# Helps when one class (e.g., Prostate: 31 samples) is much smaller than others (Breast: 193)
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    smote_available = True
    print('imbalanced-learn (SMOTE) available.')
except ImportError:
    print('imbalanced-learn not installed. Skipping SMOTE. pip install imbalanced-learn')
    smote_available = False  # flag: SMOTE won't be applied if this is False

print('\nAll core imports successful.')

In [ ]:
# Cell 2 — Paths and load data
BASE_DIR    = os.path.abspath(os.path.join(os.getcwd(), '..'))  # go one level up from Notebooks/ to Final_Solution/
RESULTS_DIR = os.path.join(BASE_DIR, 'Results')  # folder where processed CSVs live
FIGURES_DIR = os.path.join(BASE_DIR, 'Figures')  # folder where chart images will be saved

os.makedirs(RESULTS_DIR, exist_ok=True)  # create folders if they don't exist yet
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f'BASE_DIR    : {BASE_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'FIGURES_DIR : {FIGURES_DIR}')

# Load the CLR-transformed abundance table (produced by Notebook 02)
# CLR = Centered Log-Ratio: a transformation that makes compositional microbiome data suitable for ML models
# (raw fractions 0–1 are compositional and violate assumptions of most ML algorithms)
clr_path  = os.path.join(RESULTS_DIR, 'abund_combined_clr.csv')
meta_path = os.path.join(RESULTS_DIR, 'meta_combined.csv')

abund_clr = pd.read_csv(clr_path,  index_col=0)  # rows=samples, columns=genera, values=CLR scores
meta      = pd.read_csv(meta_path, index_col=0)   # cancer_type and condition per sample

# Keep only samples present in BOTH the abundance table AND the metadata
common_idx = abund_clr.index.intersection(meta.index)
abund_clr  = abund_clr.loc[common_idx]
meta       = meta.loc[common_idx]

# Sanitize column names: XGBoost rejects brackets and inequality signs in feature names
# Some genus names contain characters like "[Ruminococcus]" that break XGBoost's internal parser
import re
abund_clr.columns = [re.sub(r'[\[\]<>]', '_', col) for col in abund_clr.columns]
# re.sub replaces any of these characters: [ ] < >  with underscore _

print(f'\nLoaded CLR abundance  : {abund_clr.shape}  (samples x genera)')
print(f'Loaded metadata       : {meta.shape}')
print(f'\nMetadata columns: {list(meta.columns)}')
print(f'\ncancer_type distribution:')
print(meta['cancer_type'].value_counts())  # how many samples per cancer type
print(f'\ncondition distribution:')
print(meta['condition'].value_counts())    # Cancer vs Healthy counts

In [ ]:
# Cell 3 — Task 1 Setup: Multi-class cancer type classification
# Question: Can a machine learning model figure out WHICH CANCER TYPE a patient has, just from their microbiome?
# This is a 3-class problem: Colorectal (0), Breast (1), or Prostate (2)
# We only use cancer patients here — healthy controls don't have a cancer type to predict

cancer_mask  = meta['condition'] == 'Cancer'  # True/False mask: True for cancer rows
X_cancer     = abund_clr.loc[cancer_mask].copy()      # feature matrix: CLR abundance (inputs to model)
y_cancer_raw = meta.loc[cancer_mask, 'cancer_type'].copy()  # labels: "Breast", "Colorectal", "Prostate"

print('=== Task 1: Multi-class Cancer Type Classification ===')
print(f'Cancer-only samples  : {X_cancer.shape[0]}')     # 347 total cancer samples
print(f'Number of features   : {X_cancer.shape[1]}')     # 259 bacterial genera as features
print(f'\nClass distribution:')
print(y_cancer_raw.value_counts())  # Breast: 193, Colorectal: 123, Prostate: 31

# LabelEncoder converts text labels to integers: Breast→0, Colorectal→1, Prostate→2
# ML models work with numbers, not strings
le = LabelEncoder()
y_cancer = le.fit_transform(y_cancer_raw)  # transform labels to [0, 1, 2] array

print(f'\nLabel encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} -> {cls}')  # show the mapping for clarity

# Split into training (80%) and test (20%) sets
# stratify=y_cancer ensures each class's proportion is the same in train and test
# (important because Prostate has very few samples — we don't want all Prostate in one set)
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer,
    test_size=0.20,       # 20% held out for final evaluation
    random_state=42,      # fixed seed for reproducibility
    stratify=y_cancer     # keep class proportions balanced in both splits
)

print(f'\nTrain set  : {X_train.shape[0]} samples')  # 277 samples used for training
print(f'Test set   : {X_test.shape[0]}  samples')   # 70 samples used for final testing
print(f'Train class distribution: {dict(zip(le.classes_, np.bincount(y_train)))}')
print(f'Test  class distribution: {dict(zip(le.classes_, np.bincount(y_test)))}')

In [ ]:
# Cell 4 — Define models
# We compare 4 different classification algorithms to see which works best on microbiome data
# Each has different strengths and assumptions about the data

models = {
    # Random Forest: builds 200 independent decision trees, then votes on the answer
    # class_weight='balanced': gives more importance to rare classes (Prostate: 31 samples)
    # n_jobs=-1: use all CPU cores to train trees in parallel (faster)
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1
    ),

    # Logistic Regression with L1 regularization (Lasso): a linear model that can set
    # unimportant feature weights to exactly 0 (feature selection built-in)
    # C=0.1: strong regularization — keeps only the most important bacteria
    'Logistic Regression (L1)': LogisticRegression(
        penalty='l1', solver='liblinear', C=0.1, random_state=42,
        class_weight='balanced', max_iter=1000
    ),

    # SVM (RBF kernel): Support Vector Machine — finds the widest possible margin between classes
    # RBF = Radial Basis Function: allows non-linear boundaries in high-dimensional space
    # probability=True: needed to compute AUC (probability estimates, not just class labels)
    'SVM (RBF)': SVC(
        kernel='rbf', C=1.0, probability=True, random_state=42, class_weight='balanced'
    ),
}

# XGBoost: gradient-boosted trees — builds trees one at a time, each fixing the previous one's errors
# Only added if xgboost is installed (checked in Cell 1)
if xgb_available:
    models['XGBoost'] = XGBClassifier(
        n_estimators=200, random_state=42, eval_metric='mlogloss',
        use_label_encoder=False, n_jobs=-1
    )

print('Models defined:')
for name in models:
    print(f'  - {name}')

In [ ]:
# Cell 5 — 5-fold cross-validation for each model
# Cross-validation is like taking 5 different "exams" on different portions of the data
# The data is split into 5 equal parts (folds). In each round:
#   - 4 folds are used for training, 1 fold is used for testing
#   - This repeats 5 times so every fold gets tested exactly once
# The average score across all 5 rounds is a more reliable estimate than a single train/test split

cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 5 folds, stratified
n_classes = len(le.classes_)  # = 3 (Breast, Colorectal, Prostate)

cv_results = {}  # stores performance metrics for each model

for model_name, base_model in models.items():
    print(f'\nEvaluating: {model_name} ...')

    accs, bal_accs, f1s, aucs = [], [], [], []  # collect per-fold scores

    for fold_i, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]  # training fold
        y_tr, y_val = y_train[train_idx],      y_train[val_idx]       # validation fold labels

        # Optionally apply SMOTE to the training fold to balance class sizes
        # SMOTE creates synthetic samples for minority classes so all classes have equal representation
        if smote_available and n_classes > 1:
            try:
                min_count  = min(np.bincount(y_tr))             # size of the smallest class in this fold
                k_neighbors = min(5, min_count - 1)             # SMOTE needs at least 1 neighbor
                if k_neighbors >= 1:
                    sm = SMOTE(random_state=42, k_neighbors=k_neighbors)
                    X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)  # generate synthetic samples
                else:
                    X_tr_res, y_tr_res = X_tr, y_tr  # skip SMOTE if too few samples
            except Exception:
                X_tr_res, y_tr_res = X_tr, y_tr  # fall back to original data if SMOTE fails
        else:
            X_tr_res, y_tr_res = X_tr, y_tr  # no SMOTE available — use data as-is

        # SVM and Logistic Regression need features scaled to zero mean, unit variance
        # Wrap them in a Pipeline: StandardScaler → model (scaling is applied inside CV, not outside)
        if model_name in ['SVM (RBF)', 'Logistic Regression (L1)']:
            pipeline = Pipeline([
                ('scaler', StandardScaler()),                              # z-score scaling
                ('clf', base_model.__class__(**base_model.get_params()))   # fresh copy of the model
            ])
        else:
            pipeline = base_model.__class__(**base_model.get_params())  # RF and XGB don't need scaling

        pipeline.fit(X_tr_res, y_tr_res)    # train on this fold's training data
        y_pred = pipeline.predict(X_val)    # predict on this fold's validation data

        # Compute performance metrics for this fold
        accs.append(accuracy_score(y_val, y_pred))                              # fraction of correct predictions
        bal_accs.append(balanced_accuracy_score(y_val, y_pred))                 # accuracy adjusted for class imbalance
        f1s.append(f1_score(y_val, y_pred, average='macro', zero_division=0))  # F1 averaged equally across all classes

        # AUC (Area Under the ROC Curve): 1.0 = perfect, 0.5 = random guessing
        # OVR = One-vs-Rest: for each class, compute AUC treating it as "class vs all others"
        if hasattr(pipeline, 'predict_proba'):
            y_prob = pipeline.predict_proba(X_val)  # probability estimates per class
        elif hasattr(pipeline, 'decision_function'):
            y_prob = pipeline.decision_function(X_val)  # decision scores if no probabilities
        else:
            y_prob = None

        if y_prob is not None and len(np.unique(y_val)) == n_classes:
            try:
                auc_val = roc_auc_score(y_val, y_prob, multi_class='ovr', average='macro')
            except Exception:
                auc_val = np.nan
        else:
            auc_val = np.nan
        aucs.append(auc_val)

    # Average metrics across all 5 folds
    cv_results[model_name] = {
        'accuracy'         : np.nanmean(accs),      # mean accuracy across folds
        'balanced_accuracy': np.nanmean(bal_accs),
        'macro_f1'         : np.nanmean(f1s),
        'macro_auc'        : np.nanmean(aucs),       # primary metric for model selection
        'accuracy_std'     : np.nanstd(accs),        # variability across folds (lower = more stable)
        'macro_auc_std'    : np.nanstd(aucs),
    }

    r = cv_results[model_name]
    print(f'  Accuracy        : {r["accuracy"]:.4f} ± {r["accuracy_std"]:.4f}')
    print(f'  Balanced Acc    : {r["balanced_accuracy"]:.4f}')
    print(f'  Macro F1        : {r["macro_f1"]:.4f}')
    print(f'  Macro AUC (OVR) : {r["macro_auc"]:.4f} ± {r["macro_auc_std"]:.4f}')

print('\n=== 5-Fold CV Complete ===')

In [ ]:
# Cell 6 — Train best model on full train set, evaluate on test set
# Now that we know which model performs best in CV, we train it on ALL training data
# and evaluate ONCE on the held-out test set (the test set was never seen during CV)

# Find the model with the highest average macro-AUC across all 5 CV folds
best_model_name = max(cv_results, key=lambda k: cv_results[k]['macro_auc'])
print(f'Best model (by macro-AUC): {best_model_name}')     # SVM (RBF) wins with AUC=1.0000
print(f'CV macro-AUC: {cv_results[best_model_name]["macro_auc"]:.4f}')

# Build the final pipeline for the best model
base = models[best_model_name]
if best_model_name in ['SVM (RBF)', 'Logistic Regression (L1)']:
    # SVM and LR need scaled features — wrap in Pipeline: scaler then model
    best_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', base.__class__(**base.get_params()))  # fresh copy with same hyperparameters
    ])
else:
    best_pipeline = base.__class__(**base.get_params())  # RF/XGB: no scaling needed

# Optionally apply SMOTE to the FULL training set before training the final model
if smote_available:
    try:
        min_count   = min(np.bincount(y_train))
        k_neighbors = min(5, min_count - 1)
        if k_neighbors >= 1:
            sm = SMOTE(random_state=42, k_neighbors=k_neighbors)
            X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
        else:
            X_train_res, y_train_res = X_train.values, y_train
    except Exception:
        X_train_res, y_train_res = X_train.values, y_train
else:
    X_train_res, y_train_res = X_train.values, y_train  # use original data (no SMOTE)

best_pipeline.fit(X_train_res, y_train_res)  # train the best model on all 277 training samples

# Predict on the held-out test set (70 samples, never seen during training or CV)
y_pred_test = best_pipeline.predict(X_test)

# Get probability estimates if the model supports them (needed for AUC)
if hasattr(best_pipeline, 'predict_proba'):
    y_prob_test = best_pipeline.predict_proba(X_test)  # shape: (70, 3) — one probability per class
else:
    y_prob_test = None

# Compute final performance metrics on the test set
test_acc     = accuracy_score(y_test, y_pred_test)                              # fraction correct
test_bal_acc = balanced_accuracy_score(y_test, y_pred_test)                     # balanced version
test_f1      = f1_score(y_test, y_pred_test, average='macro', zero_division=0) # macro F1

if y_prob_test is not None:
    try:
        test_auc = roc_auc_score(y_test, y_prob_test, multi_class='ovr', average='macro')
    except Exception:
        test_auc = np.nan
else:
    test_auc = np.nan

print(f'\n=== Test Set Performance ({best_model_name}) ===')
print(f'  Accuracy        : {test_acc:.4f}')     # 1.0000 = perfect (100% correct on test set)
print(f'  Balanced Acc    : {test_bal_acc:.4f}')
print(f'  Macro F1        : {test_f1:.4f}')
print(f'  Macro AUC (OVR) : {test_auc:.4f}')
print(f'\nClassification Report:')
# Shows precision, recall, F1-score per class — useful to see if any class was harder to predict
print(classification_report(y_test, y_pred_test, target_names=le.classes_, zero_division=0))

# Save the trained model to disk so it can be reused without retraining
model_path = os.path.join(RESULTS_DIR, 'best_model_multiclass.pkl')
joblib.dump(best_pipeline, model_path)  # .pkl = pickle file = serialized Python object
print(f'\nModel saved to: {model_path}')

In [ ]:
# Cell 7 — Figure 9: ROC curves (multi-class OVR)
# ROC = Receiver Operating Characteristic curve
# For each class: plots True Positive Rate vs False Positive Rate at different thresholds
# AUC = Area Under the Curve: 1.0 = perfect classifier, 0.5 = random guessing
# OVR = One vs Rest: e.g., "Breast vs (Colorectal + Prostate)"

CLASS_COLORS = {
    'Colorectal': '#2196F3',  # blue
    'Breast'    : '#E91E63',  # pink
    'Prostate'  : '#4CAF50',  # green
}
MACRO_COLOR = 'black'

fig, ax = plt.subplots(figsize=(8, 6))

if y_prob_test is not None:
    # label_binarize converts e.g. [0, 1, 2] → [[1,0,0], [0,1,0], [0,0,1]]
    # Needed to compute per-class ROC curves
    classes_list = list(le.classes_)
    y_test_bin   = label_binarize(y_test, classes=list(range(len(classes_list))))

    all_fpr, all_tpr = [], []  # collect FPR/TPR arrays for macro-average

    for i, cls_name in enumerate(classes_list):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_test[:, i])  # ROC for this class
        roc_auc_i   = auc(fpr, tpr)                                     # AUC for this class
        color = CLASS_COLORS.get(cls_name, f'C{i}')
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{cls_name} (AUC = {roc_auc_i:.3f})')
        all_fpr.append(fpr)
        all_tpr.append(tpr)

    # Compute macro-average ROC: interpolate all per-class curves onto the same FPR grid,
    # then average the TPR values at each FPR threshold
    all_fpr_cat = np.unique(np.concatenate(all_fpr))  # merged FPR grid from all classes
    mean_tpr    = np.zeros_like(all_fpr_cat)
    for fpr_i, tpr_i in zip(all_fpr, all_tpr):
        mean_tpr += np.interp(all_fpr_cat, fpr_i, tpr_i)  # interpolate this curve onto shared grid
    mean_tpr /= len(classes_list)  # average over 3 classes
    macro_auc_val = auc(all_fpr_cat, mean_tpr)
    ax.plot(all_fpr_cat, mean_tpr, color=MACRO_COLOR, linewidth=2.5, linestyle='--',
            label=f'Macro Average (AUC = {macro_auc_val:.3f})')  # dashed black = overall performance
else:
    ax.text(0.5, 0.5, 'Probability estimates not available for this model.',
            ha='center', va='center', transform=ax.transAxes)

# Diagonal dashed line = random classifier (AUC = 0.5)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random (AUC = 0.500)')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)   # FPR = fraction of negatives incorrectly classified as positive
ax.set_ylabel('True Positive Rate', fontsize=12)    # TPR = fraction of positives correctly identified (Sensitivity)
ax.set_title(f'ROC Curves — {best_model_name} (Multi-class OVR)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
sns.despine(ax=ax)
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, 'fig09_roc_curves.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'Saved: {fig_path}')

In [ ]:
# Cell 8 — Figure 10: Confusion matrix
# A confusion matrix shows what the model predicted vs what the true label was
# Rows = true labels, Columns = predicted labels
# Diagonal cells = correct predictions (we want large numbers here)
# Off-diagonal cells = errors (e.g., predicted "Breast" when it was actually "Colorectal")

cm = confusion_matrix(y_test, y_pred_test)  # compute the N×N matrix of counts
class_labels = le.classes_                  # ['Breast', 'Colorectal', 'Prostate']

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,              # print the count number inside each cell
    fmt='d',                 # format as integer (not decimal)
    cmap='Blues',            # light blue = few, dark blue = many
    xticklabels=class_labels,  # column headers = predicted class names
    yticklabels=class_labels,  # row headers = true class names
    linewidths=0.5,          # thin lines between cells
    ax=ax
)
ax.set_xlabel('Predicted Label', fontsize=12)  # what the model said
ax.set_ylabel('True Label', fontsize=12)       # what it actually was
ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, 'fig10_confusion_matrix.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'Saved: {fig_path}')

print(f'\nConfusion Matrix:')
cm_df = pd.DataFrame(cm, index=class_labels, columns=class_labels)
print(cm_df)  # all 70 test samples correctly classified — all off-diagonal cells are 0

In [ ]:
# Cell 9 — Model comparison table
# Summarize all 4 models' cross-validation performance in a single sorted table
# This lets us see at a glance which model wins on every metric

comparison_rows = []
for model_name, metrics in cv_results.items():
    row = {
        'Model'                : model_name,
        'CV_Accuracy'          : round(metrics['accuracy'], 4),           # average accuracy across 5 folds
        'CV_Accuracy_Std'      : round(metrics['accuracy_std'], 4),       # variability (lower = more stable)
        'CV_Balanced_Accuracy' : round(metrics['balanced_accuracy'], 4),  # accuracy adjusted for class imbalance
        'CV_Macro_F1'          : round(metrics['macro_f1'], 4),           # F1 score averaged equally over all classes
        'CV_Macro_AUC'         : round(metrics['macro_auc'], 4),          # AUC averaged over all classes (primary metric)
        'CV_Macro_AUC_Std'     : round(metrics['macro_auc_std'], 4),      # AUC variability across folds
        'Best_Model'           : 'YES' if model_name == best_model_name else ''  # flag the winner
    }
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values('CV_Macro_AUC', ascending=False).reset_index(drop=True)
# Sorted so the best model appears at the top

csv_path = os.path.join(RESULTS_DIR, 'model_comparison.csv')
comparison_df.to_csv(csv_path, index=False)  # save full table for the paper supplement
print(f'Saved: {csv_path}')
print(f'\nModel Comparison (sorted by CV Macro AUC):')
print(comparison_df.to_string(index=False))  # print without row numbers for clean display

In [ ]:
# Cell 10 — Task 2: Binary classification per cancer type (Cancer vs Healthy)
# Question: Can we tell cancer patients apart from healthy people using microbiome data?
# This is separate from Task 1 — here we ask about Disease vs No Disease (not Which Disease)
# We use only Colorectal and Breast (Prostate has no healthy controls in this dataset)

binary_cancer_types = ['Colorectal', 'Breast']  # only two types have both Cancer and Healthy samples
binary_results      = []  # will store performance metrics for both cancer types

fig, axes = plt.subplots(1, 2, figsize=(14, 6))  # two side-by-side ROC plots

for ax, ct in zip(axes, binary_cancer_types):
    print(f'\n=== Binary Classification: {ct} (Cancer vs Healthy) ===')

    mask = meta['cancer_type'] == ct        # select rows for this cancer type
    X_ct = abund_clr.loc[mask].copy()      # feature matrix for this cancer type
    y_ct = (meta.loc[mask, 'condition'] == 'Cancer').astype(int)  # Cancer=1, Healthy=0

    print(f'  Total samples : {X_ct.shape[0]}')
    print(f'  Cancer        : {y_ct.sum()}')
    print(f'  Healthy       : {(y_ct == 0).sum()}')

    if y_ct.nunique() < 2:
        print(f'  SKIP: only one class present for {ct}')
        ax.set_title(f'{ct}: No healthy controls')
        continue  # skip if only one class exists (can't classify)

    # Split into 80% train, 20% test — stratified so both groups are represented in each split
    X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
        X_ct, y_ct, test_size=0.20, random_state=42, stratify=y_ct
    )

    # 5-fold cross-validation using Random Forest for binary classification
    cv_b = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rf_b = RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1
    )

    fold_aucs = []  # AUC per fold
    for tr_idx, val_idx in cv_b.split(X_tr_b, y_tr_b):
        Xf_tr, Xf_val = X_tr_b.iloc[tr_idx], X_tr_b.iloc[val_idx]
        yf_tr, yf_val = y_tr_b.iloc[tr_idx], y_tr_b.iloc[val_idx]

        # Optionally apply SMOTE inside each fold
        if smote_available:
            try:
                min_c = min(np.bincount(yf_tr.values))
                k_n   = min(5, min_c - 1)
                if k_n >= 1:
                    sm = SMOTE(random_state=42, k_neighbors=k_n)
                    Xf_tr, yf_tr = sm.fit_resample(Xf_tr, yf_tr)
            except Exception:
                pass  # use original data if SMOTE fails

        rf_b.fit(Xf_tr, yf_tr)
        y_prob_f = rf_b.predict_proba(Xf_val)[:, 1]  # probability of Cancer (class 1)
        try:
            fold_aucs.append(roc_auc_score(yf_val, y_prob_f))  # AUC for this fold
        except Exception:
            pass

    cv_auc_mean = np.mean(fold_aucs) if fold_aucs else np.nan
    cv_auc_std  = np.std(fold_aucs)  if fold_aucs else np.nan
    print(f'  CV AUC (5-fold): {cv_auc_mean:.4f} ± {cv_auc_std:.4f}')

    # Train final model on all training data and evaluate on test set
    rf_b_final = RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1
    )

    X_tr_fit, y_tr_fit = X_tr_b, y_tr_b  # starting point for training data
    if smote_available:
        try:
            min_c = min(np.bincount(y_tr_b.values))
            k_n   = min(5, min_c - 1)
            if k_n >= 1:
                sm = SMOTE(random_state=42, k_neighbors=k_n)
                X_tr_fit, y_tr_fit = sm.fit_resample(X_tr_b, y_tr_b)
        except Exception:
            pass

    rf_b_final.fit(X_tr_fit, y_tr_fit)       # train on (possibly SMOTE'd) training set
    y_prob_test_b = rf_b_final.predict_proba(X_te_b)[:, 1]  # cancer probability on test set
    y_pred_test_b = rf_b_final.predict(X_te_b)              # hard class predictions on test set

    # Compute ROC curve (series of TPR, FPR at different probability thresholds)
    try:
        fpr_b, tpr_b, _ = roc_curve(y_te_b, y_prob_test_b)
        test_auc_b       = auc(fpr_b, tpr_b)
    except Exception:
        fpr_b, tpr_b, test_auc_b = [0, 1], [0, 1], np.nan

    test_acc_b     = accuracy_score(y_te_b, y_pred_test_b)
    test_f1_b      = f1_score(y_te_b, y_pred_test_b, zero_division=0)
    test_bal_acc_b = balanced_accuracy_score(y_te_b, y_pred_test_b)

    print(f'  Test Accuracy        : {test_acc_b:.4f}')
    print(f'  Test Balanced Acc    : {test_bal_acc_b:.4f}')
    print(f'  Test Macro F1        : {test_f1_b:.4f}')
    print(f'  Test AUC             : {test_auc_b:.4f}')  # Colorectal: 0.692, Breast: 0.650

    binary_results.append({
        'cancer_type'       : ct,
        'n_cancer'          : int(y_ct.sum()),
        'n_healthy'         : int((y_ct == 0).sum()),
        'cv_auc_mean'       : round(cv_auc_mean, 4),
        'cv_auc_std'        : round(cv_auc_std, 4),
        'test_accuracy'     : round(test_acc_b, 4),
        'test_balanced_acc' : round(test_bal_acc_b, 4),
        'test_f1'           : round(test_f1_b, 4),
        'test_auc'          : round(test_auc_b, 4),
    })

    # Plot ROC curve for this cancer type
    color_map = {'Colorectal': '#2196F3', 'Breast': '#E91E63'}
    ax.plot(fpr_b, tpr_b, color=color_map.get(ct, 'navy'), linewidth=2,
            label=f'ROC (AUC = {test_auc_b:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)  # random classifier line
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(f'{ct}\nCancer vs Healthy (RF)', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    sns.despine(ax=ax)

plt.suptitle('Binary Classification: Cancer vs Healthy (Random Forest)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'fig09b_binary_roc.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'\nSaved: {fig_path}')

# Save binary results table
binary_df  = pd.DataFrame(binary_results)
binary_csv = os.path.join(RESULTS_DIR, 'binary_classification_results.csv')
binary_df.to_csv(binary_csv, index=False)
print(f'Saved: {binary_csv}')
print(f'\nBinary classification results:')
print(binary_df.to_string(index=False))

In [ ]:
# Cell 11 — Feature importance (Random Forest)
# Feature importance = how much each bacterium (feature) contributed to correct predictions
# Random Forest measures this as "Mean Decrease in Impurity": on average, how much does splitting
# on this feature reduce uncertainty at each decision tree node?
# High importance = this bacterium was frequently used and useful for distinguishing cancer types

def get_rf_from_pipeline(pipeline):
    """Try to extract the Random Forest estimator from inside a Pipeline wrapper."""
    if isinstance(pipeline, Pipeline):
        clf = pipeline.named_steps.get('clf', None)  # look for the 'clf' step inside the pipeline
        if clf is not None:
            return clf
    if isinstance(pipeline, RandomForestClassifier):
        return pipeline  # already a plain RF, return directly
    return None  # not a RF or pipeline containing RF

rf_clf = get_rf_from_pipeline(best_pipeline)

if rf_clf is None or not isinstance(rf_clf, RandomForestClassifier):
    # Best model was SVM, not RF — train a separate RF just for feature importance
    print('Best model is not a Random Forest — training dedicated RF for feature importance...')
    rf_fi = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
    rf_fi.fit(X_train, y_train)  # train on the same training data used for all models
    rf_clf = rf_fi
    print('Dedicated RF trained.')
else:
    print(f'Using Random Forest from best pipeline: {best_model_name}')

# Extract the importance score for each genus (feature)
importances  = rf_clf.feature_importances_     # array of length 259 (one score per genus)
feature_names = X_cancer.columns.tolist()      # list of genus names in the same order

# Build a sorted table: most important genus at the top
fi_df = pd.DataFrame({
    'genus'      : feature_names,
    'importance' : importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

fi_df['rank'] = fi_df.index + 1  # rank 1 = most important genus overall

# Save top 50 genera by importance (used by Notebook 07/08 for SHAP and biomarker scoring)
top50  = fi_df.head(50)
fi_csv = os.path.join(RESULTS_DIR, 'rf_feature_importances.csv')
top50.to_csv(fi_csv, index=False)
print(f'Saved top 50 feature importances: {fi_csv}')

print(f'\nTop 20 genera by Random Forest importance:')
print(fi_df.head(20).to_string(index=False))  # Mahella ranks #1 with importance 0.0596

# Horizontal bar chart: longest bar = most important bacterium
top20 = fi_df.head(20).copy()

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(
    top20['genus'][::-1],       # reversed so #1 appears at the top
    top20['importance'][::-1],  # corresponding importance scores
    color='#1976D2',            # blue bars
    edgecolor='white'
)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=12)
ax.set_title('Top 20 Most Important Genera\n(Random Forest, Multi-class Cancer Type)', fontsize=13, fontweight='bold')
sns.despine(ax=ax)  # remove top and right borders
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, 'fig09c_rf_feature_importance.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'Saved: {fig_path}')

print('\n=== Notebook 06 Complete ===')
print(f'Best model            : {best_model_name}')  # SVM (RBF), AUC = 1.0000
print(f'Model saved to        : {os.path.join(RESULTS_DIR, "best_model_multiclass.pkl")}')
print(f'Feature importances   : {fi_csv}')
print(f'Model comparison      : {os.path.join(RESULTS_DIR, "model_comparison.csv")}')
print(f'Binary results        : {os.path.join(RESULTS_DIR, "binary_classification_results.csv")}')